# Additive conservation laws of generalised Langton's ants, for all moduli at once

*For a colleague who has not seen the work.* An n-colour ant turns by `rule[c]` on colour `c`, advances the
colour to `(c+1) % n` and steps. We ask for **exact** laws `V = a x + b y + e t + Phi(h, class) + sum_cells w(class, colour)`
that no step can change (mod some `m`). Each law is a proof, not a statistic: it is the solution of a finite
integer-linear system (every local transition), solved for all `m` at once by Smith normal form.
Predictions were registered in the README before any code existed (commit `db7eb40`).

## Provenance

In [1]:
import json, hashlib, csv, pathlib
from collections import Counter
R = pathlib.Path('results') if pathlib.Path('results').exists() else pathlib.Path('experiments/archive/2026-10-invariant-mining-snf/results')
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()[:16]
prov = json.load(open(R / 'census_provenance.json'))
print('census command :', prov['command'])
print('census git head:', prov['git_head'])
print('lawmod.py sha  :', prov['lawmod_sha256'][:16])
for name in ['fingerprints.csv','universal.csv','p4.csv','by_counts.csv','summary.json','oracle_summary.json','crosscheck_peer.json','highways_tilefree.json']:
    print(f'{name:26s} sha256[:16] = {sha(R/name)}')

census command : experiments/2026-10-22-invariant-mining-snf/scripts/census.py --nmax 8
census git head: 53778d11804917ece5a1287635e92f16059e72e1
lawmod.py sha  : 9aea69c1a5953d6d
fingerprints.csv           sha256[:16] = d0a6b0228f87effc
universal.csv              sha256[:16] = b764e1339b7f3b9e
p4.csv                     sha256[:16] = 6f371fc3c21efef2
by_counts.csv              sha256[:16] = 00e02e64672b4085
summary.json               sha256[:16] = f08cbcda1a1dda85
oracle_summary.json        sha256[:16] = 849eecf83b4ffa69
crosscheck_peer.json       sha256[:16] = f0d33debe45278a3
highways_tilefree.json     sha256[:16] = 79eddd7bb17b783e


## Assumptions and what a law means

* One ant on the square lattice, blank (colour 0) background, finite-support field; with an infinite non-blank
  background read the field sum as the finite change from the initial field.
* Families: **A0** (no class, `Phi = d*h`, the other agent's ansatz), **F1** (no class, any `Phi(h)`),
  **F2** (class `(x+y) mod 2`, only states with `h = x+y` mod 2), **F3** (class `(x mod 2, y mod 2)`, same manifold).
  The manifold `h + x + y` even is itself an invariant, and the blank start lies on it.
* *Universal* laws hold for every rule of that length; the interesting part of a rule is its law group modulo the
  universal one (summarised by Smith invariant factors).

## P1: the law group of a rule depends only on (n, #R)

In [2]:
S = json.load(open(R / 'summary.json'))
print('rules per family            :', S['P1_rules_per_family'])
print('(family,n,#R) with >1 fingerprint:', S['P1_exceptions'])
print('post hoc: groups by (n, +-D mod 4, trivial):', S['P1b_posthoc_groups'], ' with >1 fingerprint:', S['P1b_posthoc_groups_with_more_than_one_fingerprint'])
print('differences under complement / reverse / rotations:', S['P1_symmetry_differences'])
print('distinct fingerprints per (family, n):')
for k, v in S['distinct_fingerprints_per_family_n'].items():
    if k.startswith('F1') or k.startswith('F3'): print(f'  {k}: {v}')

rules per family            : 508
(family,n,#R) with >1 fingerprint: 0
post hoc: groups by (n, +-D mod 4, trivial): 68  with >1 fingerprint: 0
differences under complement / reverse / rotations: {'complement': 0, 'reverse': 0, 'rotations': 0, 'checked': 2032}
distinct fingerprints per (family, n):
  F1 n=2: 2
  F1 n=3: 2
  F1 n=4: 3
  F1 n=5: 2
  F1 n=6: 3
  F1 n=7: 2
  F1 n=8: 3
  F3 n=2: 2
  F3 n=3: 2
  F3 n=4: 3
  F3 n=5: 2
  F3 n=6: 3
  F3 n=7: 2
  F3 n=8: 3


Reading: `P1_exceptions` = 0 means no rule of length <= 8 has a law group different from the other rules with the same
number of R. In particular `RRLL` and `RLRL` are indistinguishable to every law in A0-F3: these invariants are *blind to
the order of the colours*. That follows from a short argument (README, "Derivation behind P1"): a cell potential
cannot see the heading, so each colour's net contribution is a constant `E_R` or `E_L`, and only
`#R*E_R + #L*E_L = 0` survives the trip round the colour cycle.

## P3, P6, P7: universal laws (true of every rule of the length)

In [3]:
print('P3 (F1 universal orders; observed, predicted):')
for k, v in S['P3_cells'].items(): print(f'  {k:12s} {v}')
print('P3 all as predicted:', S['P3_all_as_predicted'])
print()
print('Universal group order at modulus m = n  (P6, P7):')
for n, d in S['P6_orders_at_m_eq_n'].items(): print(f'  n={n}: {d}')
print('P6 F2 > F1 for every n:', S['P6_F2_gt_F1_all_n'])
print('P7 F3 > F2 for some n :', S['P7_F3_gt_F2_some_n'], S['P7_ns_where_F3_gt_F2'])

P3 (F1 universal orders; observed, predicted):
  n=2 m=2      [8, 8]
  n=2 m=3      [1, 1]
  n=2 m=5      [1, 1]
  n=2 m=7      [1, 1]
  n=3 m=2      [4, 4]
  n=3 m=3      [3, 3]
  n=3 m=5      [1, 1]
  n=3 m=7      [1, 1]
  n=4 m=2      [8, 8]
  n=4 m=3      [1, 1]
  n=4 m=5      [1, 1]
  n=4 m=7      [1, 1]
  n=5 m=2      [4, 4]
  n=5 m=3      [1, 1]
  n=5 m=5      [5, 5]
  n=5 m=7      [1, 1]
  n=6 m=2      [8, 8]
  n=6 m=3      [3, 3]
  n=6 m=5      [1, 1]
  n=6 m=7      [1, 1]
  n=7 m=2      [4, 4]
  n=7 m=3      [1, 1]
  n=7 m=5      [1, 1]
  n=7 m=7      [7, 7]
  n=8 m=2      [8, 8]
  n=8 m=3      [1, 1]
  n=8 m=5      [1, 1]
  n=8 m=7      [1, 1]
P3 all as predicted: True

Universal group order at modulus m = n  (P6, P7):
  n=2: {'F1': 8, 'F2': 16, 'F3': 64}
  n=3: {'F1': 3, 'F2': 9, 'F3': 81}
  n=4: {'F1': 32, 'F2': 64, 'F3': 1024}
  n=5: {'F1': 5, 'F2': 25, 'F3': 625}
  n=6: {'F1': 24, 'F2': 144, 'F3': 5184}
  n=7: {'F1': 7, 'F2': 49, 'F3': 2401}
  n=8: {'F1': 64, 'F2': 256, 

## P4: when does the tape determine the heading?

In [4]:
print('rules:', S['P4_rules'], ' with D=#R-#L = 0 mod 4:', S['P4_rules_with_D0'])
print('registered P4 (any law a=b=e=0, Phi nonconstant, mod 4 iff D=0 mod 4): mismatches =', S['P4_registered_mismatches'],
      ' D values of mismatches =', S['P4_mismatch_D_values'], ' all n even:', S['P4_mismatch_all_n_even'])
print('post hoc refinement (a law with Phi(1) odd, i.e. pins h mod 4):          mismatches =', S['P4_posthoc_mismatches'])

rules: 508  with D=#R-#L = 0 mod 4: 170
registered P4 (any law a=b=e=0, Phi nonconstant, mod 4 iff D=0 mod 4): mismatches = 170  D values of mismatches = ['2']  all n even: True
post hoc refinement (a law with Phi(1) odd, i.e. pins h mod 4):          mismatches = 0


The registered statement was too strong: for even `n` the colour clock gives `t = sum c N_c (mod 2)` and the heading parity
equals the time parity, so heading *parity* is always a function of the tape. What `D = 0 (mod 4)` buys is the full
heading mod 4 (Langton's ant: heading = number of colour-1 cells mod 4). The refinement was written after seeing the
failure and is labelled post hoc.

## P5 and P2: are the laws really laws? (oracle and cross-check)

In [5]:
O = json.load(open(R / 'oracle_summary.json'))
print('oracle: steps', O['steps'], 'starts/rule', O['starts_per_rule'], 'seed', O['seed'], 'git', O['git_head'][:10])
for k, v in O['by_family'].items(): print(f'  {k:12s} {v}')
print('law violations:', O['law_violations_total'], ' control detection rate:', O['control_detection_rate'])
X = json.load(open(R / 'crosscheck_peer.json'))
print('cross-check vs src/invariants.rs: comparisons', X['comparisons'], 'mismatches', X['mismatches'],
      '| peer files committed when run:', X['peer_files_committed'])

oracle: steps 2000 starts/rule 16 seed 20261022 git 53778d1180
  A0:control   {'lines': 1640, 'lines_with_violation': 1640, 'applications': 26240}
  A0:law       {'lines': 2280, 'lines_with_violation': 0, 'applications': 36480}
  F1:control   {'lines': 1640, 'lines_with_violation': 1640, 'applications': 26240}
  F1:law       {'lines': 2816, 'lines_with_violation': 0, 'applications': 45056}
  F2:control   {'lines': 1640, 'lines_with_violation': 1640, 'applications': 13120}
  F2:law       {'lines': 3672, 'lines_with_violation': 0, 'applications': 29376}
  F3:control   {'lines': 1640, 'lines_with_violation': 1640, 'applications': 13120}
  F3:law       {'lines': 5648, 'lines_with_violation': 0, 'applications': 45184}
law violations: 0  control detection rate: 1.0
cross-check vs src/invariants.rs: comparisons 1016 mismatches 0 | peer files committed when run: True


## P8: tile-free laws against the confirmed highways

In [6]:
H = json.load(open(R / 'highways_tilefree.json'))
print('highways:', H['n_highways'])
print('violations:', {k: v['count'] for k, v in H['violations'].items()})
print('tile-free group orders at m=2,4,8 (rules with that order):', H['tile_free_group_orders_m2_m4_m8'])
print('tile-free group equals the universal one for every rule (orders at m=2,4,8):', H['rule_tile_free_equals_universal_for_all'])

highways: 582
violations: {'F1': 0, 'F2': 0, 'F3': 0}
tile-free group orders at m=2,4,8 (rules with that order): {'F1': {'(4, 8, 8)': 582}, 'F2': {'(4, 8, 8)': 582}, 'F3': {'(8, 16, 16)': 582}}
tile-free group equals the universal one for every rule (orders at m=2,4,8): {'F1': True, 'F2': True, 'F3': True}


## What this does and does not show
* **Shows (exact, finite):** the complete law group of every rule of length <= 8 within A0-F3, its universal part, and
  that the rule-dependent part is a function of `(n, #R)`. Every generator survived 2000 oracle steps from random starts, and
  random non-laws were caught, so the checker has power.
* **Does not show:** anything about growth exponents, highway formation or chaos. Laws that couple the ant to the
  colours of neighbouring cells, pair potentials, edge flux and history dependence are outside the ansatz and are the only
  place rule order could enter an invariant. The tile-free highway check is an instrument check (no violations, nothing new
  beyond known parity facts).
* **Caveats:** the oracle checks lengths <= 6; the census covers <= 8; the claim for all `n` rests on the short argument
  above, not on a checked proof. The README's rule count (510) is a typo for 508 (2^2 + ... + 2^8). P4 as registered failed.

## Next
Build a pair-potential family with a quotient by functionals that vanish on all finite fields (otherwise group orders are
inflated), and ask whether any law there depends on the *order* of the colours.